In [1]:
"""
임상시험 데이터 FAISS 인덱싱
- Qwen3-Embedding-4B 모델로 임베딩
- 55만개 데이터를 배치 처리
- FAISS 인덱스 + 메타데이터 저장
"""

import os
import json
import time
import pickle
import numpy as np
from tqdm.notebook import tqdm
import torch
from transformers import AutoTokenizer, AutoModel
import faiss

# ============================================================
# 1. 설정
# ============================================================

# 경로 설정
DATA_PATH = "pasred_data/parsed_ctg-studies.json"
INDEX_PATH = "faiss_index"  # 저장 디렉토리
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-4B"  # 또는 Qwen3-Embedding

# 배치 설정
BATCH_SIZE = 32  # GPU 메모리에 따라 조절
MAX_LENGTH = 512

# GPU 설정
DEVICE = "cuda:1"  # 사용할 GPU


# ============================================================
# 2. 임베딩 모델 로딩
# ============================================================

def load_embedding_model():
    """Qwen Embedding 모델 로딩"""
    start = time.time()
    print("🔄 임베딩 모델 로딩 중...")
    
    tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        EMBEDDING_MODEL,
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to(DEVICE)
    model.eval()
    
    print(f"✅ 임베딩 모델 로딩 완료 ({time.time() - start:.1f}초)")
    return tokenizer, model


# ============================================================
# 3. 텍스트 추출 함수
# ============================================================

def extract_searchable_text(nct_id: str, study_data: dict) -> str:
    """
    임상시험 데이터에서 검색용 텍스트 추출
    Brief Title + Conditions + Brief Summary + Interventions
    """
    overview = study_data.get("Study Overview", {})
    
    # 주요 필드 추출
    brief_title = overview.get("Brief Title", "")
    official_title = overview.get("Official Title", "")
    conditions = overview.get("Conditions", [])
    brief_summary = overview.get("Brief Summary", "")
    study_type = overview.get("Study Type", "")
    
    # Interventions 추출
    interventions = overview.get("Interventions", [])
    intervention_texts = []
    for inv in interventions:
        inv_type = inv.get("Type", "")
        inv_name = inv.get("Name", "")
        if inv_type or inv_name:
            intervention_texts.append(f"{inv_type}: {inv_name}")
    
    # 검색용 텍스트 조합
    text_parts = [
        f"Title: {brief_title}",
        f"Conditions: {', '.join(conditions) if conditions else 'N/A'}",
        f"Study Type: {study_type}",
        f"Interventions: {', '.join(intervention_texts) if intervention_texts else 'N/A'}",
        f"Summary: {brief_summary[:500]}"  # 요약은 500자로 제한
    ]
    
    return " | ".join(text_parts)


# ============================================================
# 4. 배치 임베딩 함수
# ============================================================

def get_embeddings(texts: list, tokenizer, model) -> np.ndarray:
    """텍스트 리스트를 임베딩 벡터로 변환"""
    
    with torch.no_grad():
        inputs = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(DEVICE)
        
        outputs = model(**inputs)
        
        # Mean pooling
        attention_mask = inputs["attention_mask"]
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        
        # Normalize
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        
    return embeddings.cpu().numpy().astype('float32')


# ============================================================
# 5. 메인 인덱싱 함수
# ============================================================

def build_faiss_index():
    """FAISS 인덱스 구축"""
    
    # 저장 디렉토리 생성
    os.makedirs(INDEX_PATH, exist_ok=True)
    
    # 모델 로딩
    tokenizer, model = load_embedding_model()
    
    # 데이터 로딩
    print(f"🔄 데이터 로딩 중: {DATA_PATH}")
    start = time.time()
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"✅ 데이터 로딩 완료: {len(data):,}개 ({time.time() - start:.1f}초)")
    
    # 텍스트 추출 및 메타데이터 준비
    print("🔄 검색용 텍스트 추출 중...")
    nct_ids = []
    texts = []
    
    for nct_id, study_data in tqdm(data.items(), desc="텍스트 추출"):
        text = extract_searchable_text(nct_id, study_data)
        if text.strip():
            nct_ids.append(nct_id)
            texts.append(text)
    
    print(f"✅ 텍스트 추출 완료: {len(texts):,}개")
    
    # 배치 임베딩
    print("🔄 임베딩 생성 중...")
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="임베딩"):
        batch_texts = texts[i:i + BATCH_SIZE]
        batch_embeddings = get_embeddings(batch_texts, tokenizer, model)
        all_embeddings.append(batch_embeddings)
        
        # GPU 메모리 정리 (주기적으로)
        if i % (BATCH_SIZE * 100) == 0:
            torch.cuda.empty_cache()
    
    embeddings = np.vstack(all_embeddings)
    print(f"✅ 임베딩 완료: shape={embeddings.shape}")
    
    # FAISS 인덱스 생성
    print("🔄 FAISS 인덱스 생성 중...")
    dimension = embeddings.shape[1]
    
    # Inner Product (코사인 유사도, 정규화된 벡터)
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    
    print(f"✅ FAISS 인덱스 생성 완료: {index.ntotal:,}개 벡터")
    
    # 저장
    print("🔄 저장 중...")
    
    # FAISS 인덱스 저장
    faiss.write_index(index, os.path.join(INDEX_PATH, "clinical_trials.index"))
    
    # 메타데이터 저장 (NCT ID 리스트)
    with open(os.path.join(INDEX_PATH, "nct_ids.pkl"), 'wb') as f:
        pickle.dump(nct_ids, f)
    
    # 원본 데이터도 저장 (검색 후 Few-shot에 사용)
    with open(os.path.join(INDEX_PATH, "study_data.pkl"), 'wb') as f:
        pickle.dump(data, f)
    
    print(f"✅ 저장 완료: {INDEX_PATH}/")
    print(f"   - clinical_trials.index ({index.ntotal:,} vectors)")
    print(f"   - nct_ids.pkl ({len(nct_ids):,} IDs)")
    print(f"   - study_data.pkl")
    
    # GPU 메모리 해제
    del model
    del tokenizer
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    
    return index, nct_ids, data


# ============================================================
# 6. 실행
# ============================================================

if __name__ == "__main__":
    total_start = time.time()
    
    print("\n" + "="*70)
    print("🔬 임상시험 FAISS 인덱싱 시작")
    print("="*70 + "\n")
    
    index, nct_ids, data = build_faiss_index()
    
    print("\n" + "="*70)
    print(f"✅ 전체 완료! (총 {time.time() - total_start:.1f}초)")
    print("="*70)

/data/home/bmi-lab/anaconda3/envs/jy3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🔬 임상시험 FAISS 인덱싱 시작

🔄 임베딩 모델 로딩 중...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.44it/s]


✅ 임베딩 모델 로딩 완료 (868.4초)
🔄 데이터 로딩 중: pasred_data/parsed_ctg-studies.json
✅ 데이터 로딩 완료: 557,387개 (60.6초)
🔄 검색용 텍스트 추출 중...


텍스트 추출: 100%|██████████| 557387/557387 [00:02<00:00, 266220.14it/s]


✅ 텍스트 추출 완료: 557,387개
🔄 임베딩 생성 중...


임베딩: 100%|██████████| 17419/17419 [2:39:31<00:00,  1.82it/s]  


✅ 임베딩 완료: shape=(557387, 2560)
🔄 FAISS 인덱스 생성 중...
✅ FAISS 인덱스 생성 완료: 557,387개 벡터
🔄 저장 중...
✅ 저장 완료: faiss_index/
   - clinical_trials.index (557,387 vectors)
   - nct_ids.pkl (557,387 IDs)
   - study_data.pkl

✅ 전체 완료! (총 10540.3초)
